### Imports and data loading

In [18]:
import pandas as pd
from transformers import AutoModel, AutoTokenizer
import torch
import torch.nn.functional as F
import random
from torch.utils.data import Dataset, DataLoader
from torch import nn
from torch.optim import AdamW

train_data = pd.read_csv("train_v3/train_data.csv")

raw_questions = list(pd.read_csv("train_v3/raw_questions.csv")['question'])

In [2]:
similar = train_data[['question1', 'question2']]
similar_q1 = list(similar['question1'])
similar_q2 = list(similar['question2'])

### Finetune

In [3]:

model = AutoModel.from_pretrained("train_v3//my_custom_bert_model")
tokenizer = AutoTokenizer.from_pretrained("train_v3//my_custom_bert_model")

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

In [4]:
tokenized_similar_q1 = tokenizer(similar_q1, return_tensors="pt", padding=True, truncation=True)
tokenized_similar_q2 = tokenizer(similar_q2, return_tensors="pt", padding=True, truncation=True)
tokenized_similar_q1

{'input_ids': tensor([[ 101, 2129, 2064,  ...,    0,    0,    0],
        [ 101, 2054, 1005,  ...,    0,    0,    0],
        [ 101, 2339, 2079,  ...,    0,    0,    0],
        ...,
        [ 101, 2079, 4176,  ...,    0,    0,    0],
        [ 101, 2129, 2079,  ...,    0,    0,    0],
        [ 101, 2003, 1996,  ...,    0,    0,    0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])}

In [5]:
similar_output_q1 = model(**tokenized_similar_q1)
similar_output_q2 = model(**tokenized_similar_q2)

similar_emb_q1 = similar_output_q1.last_hidden_state.mean(dim=1)
similar_emb_q2 = similar_output_q2.last_hidden_state.mean(dim=1)
similar_emb_q1.shape

torch.Size([500, 512])

In [6]:
cos_sims = F.cosine_similarity(similar_emb_q1, similar_emb_q2, dim=1)
cos_sims

tensor([0.9247, 0.9323, 0.9373, 0.9340, 0.9465, 0.9594, 0.9270, 0.9525, 0.9824,
        0.9848, 0.8779, 0.9822, 0.9769, 0.9853, 0.8927, 0.8900, 0.9297, 0.9592,
        0.8843, 0.9794, 0.9931, 0.9436, 0.9336, 0.9449, 0.8807, 0.8743, 0.9882,
        0.8880, 0.9864, 0.9360, 0.9233, 0.9493, 0.9851, 0.9040, 0.9867, 0.9189,
        0.9391, 0.9674, 0.9123, 0.9584, 0.9896, 0.9820, 0.8717, 0.9845, 0.9134,
        0.9689, 0.9733, 0.9036, 0.9422, 0.8879, 0.9616, 0.9318, 0.8900, 0.9307,
        0.9614, 0.9766, 0.9062, 0.9176, 0.9930, 0.9259, 0.8830, 0.9293, 0.9162,
        0.9086, 0.8871, 0.9391, 0.9275, 0.8481, 0.7722, 0.9678, 0.9524, 0.9841,
        0.9105, 0.8905, 0.9279, 0.9310, 0.8782, 0.9465, 0.9618, 0.9156, 0.9437,
        0.9539, 0.9123, 0.9240, 0.8841, 0.8881, 0.9730, 0.9452, 0.9143, 0.8937,
        0.8848, 0.8691, 0.9302, 0.8209, 0.9660, 0.9081, 0.9773, 0.9018, 0.8584,
        0.8964, 0.8890, 0.9335, 0.9632, 0.9695, 0.9432, 0.9696, 0.9347, 0.9531,
        0.9412, 0.9488, 0.9562, 0.9153, 

In [7]:
tokenized_raw = tokenizer(raw_questions, return_tensors='pt', padding=True, truncation=True)
raw_emb = model(**tokenized_raw).last_hidden_state.mean(dim=1)
emb_norm = F.normalize(raw_emb, dim=1)
sim_matrix = emb_norm @ emb_norm.T
sim_matrix

tensor([[1.0000, 0.6914, 0.7667,  ..., 0.8246, 0.7935, 0.8968],
        [0.6914, 1.0000, 0.6712,  ..., 0.7270, 0.7018, 0.6622],
        [0.7667, 0.6712, 1.0000,  ..., 0.7587, 0.8002, 0.7526],
        ...,
        [0.8246, 0.7270, 0.7587,  ..., 1.0000, 0.7853, 0.7536],
        [0.7935, 0.7018, 0.8002,  ..., 0.7853, 1.0000, 0.7569],
        [0.8968, 0.6622, 0.7526,  ..., 0.7536, 0.7569, 1.0000]],
       grad_fn=<MmBackward0>)

In [8]:
sim_matrix.fill_diagonal_(-float('inf'))
n = 1000
values, indices = torch.topk(sim_matrix.reshape(-1), k=n)
values

tensor([1.0000, 1.0000, 0.9986, 0.9986, 0.9983, 0.9983, 0.9961, 0.9961, 0.9955,
        0.9955, 0.9942, 0.9942, 0.9930, 0.9930, 0.9925, 0.9925, 0.9920, 0.9920,
        0.9894, 0.9894, 0.9890, 0.9890, 0.9886, 0.9886, 0.9885, 0.9885, 0.9883,
        0.9883, 0.9882, 0.9882, 0.9880, 0.9880, 0.9879, 0.9879, 0.9876, 0.9876,
        0.9873, 0.9873, 0.9866, 0.9866, 0.9866, 0.9866, 0.9864, 0.9864, 0.9863,
        0.9863, 0.9856, 0.9856, 0.9854, 0.9854, 0.9849, 0.9849, 0.9845, 0.9845,
        0.9842, 0.9842, 0.9834, 0.9834, 0.9830, 0.9830, 0.9830, 0.9830, 0.9829,
        0.9829, 0.9827, 0.9827, 0.9823, 0.9823, 0.9820, 0.9820, 0.9818, 0.9818,
        0.9818, 0.9818, 0.9810, 0.9810, 0.9810, 0.9810, 0.9805, 0.9805, 0.9795,
        0.9795, 0.9792, 0.9792, 0.9792, 0.9792, 0.9787, 0.9787, 0.9782, 0.9782,
        0.9778, 0.9778, 0.9768, 0.9768, 0.9765, 0.9765, 0.9763, 0.9763, 0.9763,
        0.9763, 0.9759, 0.9759, 0.9756, 0.9756, 0.9750, 0.9750, 0.9730, 0.9730,
        0.9726, 0.9726, 0.9723, 0.9723, 

In [9]:
N = 1000
rows = indices // N
cols = indices % N
top_pairs = list(zip(rows.detach().numpy(), cols.detach().numpy(), values.detach().numpy()))
top_pairs[0]

(np.int64(180), np.int64(551), np.float32(1.0))

In [10]:
i=700
print(raw_questions[top_pairs[i][0]], raw_questions[top_pairs[i][1]], top_pairs[i][2])

What is the best way to learn about stock market? What's the best way to quit meth? 0.9153552


In [11]:
for i in range(10):
    print(raw_questions[top_pairs[i][0]], raw_questions[top_pairs[i][1]], top_pairs[i][2])

How does a top-less shoot get done in India for models & actresses? How does a top-less shoot get done in India for models & actresses? 1.0
How does a top-less shoot get done in India for models & actresses? How does a top-less shoot get done in India for models & actresses? 1.0
What do you anticipate will be the key factors in Meaningful Use Stage 3? What do you anticipate will be the key factors in Meaningful Use Stage 2? 0.99863714
What do you anticipate will be the key factors in Meaningful Use Stage 2? What do you anticipate will be the key factors in Meaningful Use Stage 3? 0.99863714
How does the HP OfficeJet 4620 Airprint compare to the HP LaserJet Enterprise M506x? How does the HP OfficeJet 4620 Airprint compare to the HP LaserJet Enterprise M506n? 0.99828726
How does the HP OfficeJet 4620 Airprint compare to the HP LaserJet Enterprise M506n? How does the HP OfficeJet 4620 Airprint compare to the HP LaserJet Enterprise M506x? 0.99828726
What are some things new employees shoul

### Training

In [12]:
pos_pairs = list(zip(similar_q1, similar_q2, [1.0] * len(similar_q1)))
neg_pairs = [(raw_questions[top_pairs[i][0]], raw_questions[top_pairs[i][1]], 0) for i in range(201, 701)]
all_pairs = pos_pairs + neg_pairs
random.shuffle(all_pairs)
len(all_pairs)

1000

In [13]:
train_val_split = int(0.9*len(all_pairs))
train_pairs, val_pairs = all_pairs[:train_val_split], all_pairs[train_val_split:]
len(train_pairs), len(val_pairs)

(900, 100)

In [41]:
class PairDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)
    
    def __getitem__(self, index):
        pair = self.pairs[index]
        return (pair[0], pair[1], torch.tensor(pair[2], dtype=torch.float))
    
def collate_fn(batch):
    q1s, q2s, labels = zip(*batch)
    tokenized_q1s = tokenizer(q1s, return_tensors='pt', padding=True, truncate=True, max_length=128) #max_length?
    tokenized_q2s = tokenizer(q2s, return_tensors='pt', padding=True, truncate=True, max_length=128)
    return tokenized_q1s, tokenized_q2s, torch.stack(labels)

train_loader = DataLoader(PairDataset(train_pairs), collate_fn=collate_fn, batch_size=64, shuffle=True)
val_loader = DataLoader(PairDataset(val_pairs), collate_fn=collate_fn, batch_size=64, shuffle=False)

In [42]:
class ContrastiveLoss(nn.Module):
    def __init__(self, margin = 0.5):
        super().__init__()
        self.margin = margin

    def forward(self, emb1, emb2, labels):
        cos_sim = F.cosine_similarity(emb1, emb2)
        loss = labels * (1-cos_sim).pow(2) + (1-labels) * F.relu(cos_sim-self.margin).pow(2)
        return loss.mean()

In [48]:
EPOCHS = 20
LR = 2e-5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)
optim = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
criterion = ContrastiveLoss()

def move(enc, device):
    return {k: v.to(device) for k,v in enc.items()}

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    for (enc1s, enc2s, labels) in train_loader:
        enc1s, enc2s, labels = move(enc1s, device), move(enc2s, device), labels.to(device)
        emb1 = model(**enc1s).last_hidden_state.mean(dim=1)
        emb2 = model(**enc2s).last_hidden_state.mean(dim=1)

        loss = criterion(emb1, emb2, labels)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0) # why?
        optim.step()
        optim.zero_grad()

        train_loss += loss.item()

    model.eval()
    val_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for (enc1s, enc2s, labels) in val_loader:
            enc1s, enc2s, labels = move(enc1s, device), move(enc2s, device), labels.to(device)
            emb1 = model(**enc1s).last_hidden_state.mean(dim=1)
            emb2 = model(**enc2s).last_hidden_state.mean(dim=1)

            val_loss += criterion(emb1, emb2, labels).item()
            preds = (F.cosine_similarity(emb1, emb2) > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
        
    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Train Loss: {train_loss/len(train_loader):.4f} | "
          f"Val Loss: {val_loss/len(val_loader):.4f} | "
          f"Val Acc: {correct/total:.3f}")

Epoch 1/20 | Train Loss: 0.0184 | Val Loss: 0.0395 | Val Acc: 0.660
Epoch 2/20 | Train Loss: 0.0169 | Val Loss: 0.0377 | Val Acc: 0.640
Epoch 3/20 | Train Loss: 0.0138 | Val Loss: 0.0348 | Val Acc: 0.680
Epoch 4/20 | Train Loss: 0.0123 | Val Loss: 0.0317 | Val Acc: 0.660
Epoch 5/20 | Train Loss: 0.0100 | Val Loss: 0.0296 | Val Acc: 0.680
Epoch 6/20 | Train Loss: 0.0093 | Val Loss: 0.0275 | Val Acc: 0.650
Epoch 7/20 | Train Loss: 0.0077 | Val Loss: 0.0247 | Val Acc: 0.740
Epoch 8/20 | Train Loss: 0.0071 | Val Loss: 0.0233 | Val Acc: 0.710
Epoch 9/20 | Train Loss: 0.0059 | Val Loss: 0.0222 | Val Acc: 0.750
Epoch 10/20 | Train Loss: 0.0053 | Val Loss: 0.0209 | Val Acc: 0.740
Epoch 11/20 | Train Loss: 0.0051 | Val Loss: 0.0202 | Val Acc: 0.790
Epoch 12/20 | Train Loss: 0.0047 | Val Loss: 0.0200 | Val Acc: 0.770
Epoch 13/20 | Train Loss: 0.0042 | Val Loss: 0.0195 | Val Acc: 0.800
Epoch 14/20 | Train Loss: 0.0039 | Val Loss: 0.0186 | Val Acc: 0.820
Epoch 15/20 | Train Loss: 0.0037 | Val Loss